In [ ]:
import os
import random
from datasets import load_dataset

# ======================================================================
# 📚 튜터의 메모: 이 데이터셋은 한국의 법률 및 공정무역 관련 지식을 
# 묻고 답하는 '질의응답(Q&A)' 형태의 데이터셋입니다.
# 🌐 의미: 학습자가 주어진 질문(instruction)과 배경 정보(input)를 
# 분석해서 정답(output)을 찾아내는 '지식 추출 및 요약' 능력을 기르는 데 사용됩니다.
# ✨ 목표: 이 데이터를 이용해 법률 지식을 전문적으로 처리하는 AI의 뇌(Brain)를 만들어 봅시다!
# ======================================================================

# --------------------- 설정 값 ----------------------
DATASET_NAME = "naenghoon/Fair-Trade-Korea-Law-Presidential"
SAMPLE_COUNT = 10 # 실습을 위해 상위 10개의 샘플만 사용합니다!
# -----------------------------------------------------


print("🌟 안녕하세요! 법률 AI의 심장부를 탐험할 준비가 되셨나요? 🚀")
print(f"📚 데이터셋 이름: {DATASET_NAME}")
print("------------------------------------------------------------------")


# 1. 데이터셋 로딩 전략: 스트리밍 모드 시도 (속도 최우선!)
dataset = None
sample_data_list = []

try:
    print("🔍 STEP 1: 데이터셋을 스트리밍 모드로 로드합니다... (가장 빠르고 효율적인 방법!)")
    # 스트리밍으로 시도! 이게 성공하면 정말 빠르다는 뜻이에요.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드 로드 완료! 데이터를 느린 왕관보다 빠르게 만날 수 있습니다.")

except Exception as e:
    print(f"⚠️ 스트리밍 로드 실패 ({e.__class__.__name__}) - 일반 로드로 전환합니다.")
    # 스트리밍 실패 시, 전체 데이터를 메모리에 로드합니다. (작은 데이터셋이라 괜찮아요!)
    try:
        dataset = load_dataset(DATASET_NAME, split='train')
        print("✅ 성공! 일반 모드 로드 완료! 이제 데이터를 마음껏 탐험해 봐요.")
    except Exception as e_fallback:
        print(f"❌ 치명적인 에러 발생: 데이터셋 로드 실패. {e_fallback}")
        exit()


# 2. 스트리밍 여부에 따라 샘플링 방식 분기 (매우 중요!)
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)인 경우
    print("\n✨ 샘플링 모드: 스트리밍 이터레이터를 사용하여 상위 K개만 가져옵니다.")
    # next()를 사용하여 샘플을 하나씩 가져옵니다.
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sample_data_list = list(sample_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    print("\n✨ 샘플링 모드: 일반 Dataset 객체를 사용하여 상위 K개만 가져옵니다.")
    sample_data_list = dataset.select(range(min(SAMPLE_COUNT, len(dataset))))


if not sample_data_list:
    print("\n🚨 아! 샘플 데이터를 가져오지 못했습니다. 다음 기회에 다시 도전해 봐요!")
    exit()


print(f"\n🎉 준비 완료! 총 {len(sample_data_list)}개의 샘플을 가져와 실습할 수 있게 되었어요.")

# ----------------------------------------------------------------------
# 🎨 실습 1: [데이터 구조 분석가] 법률 AI의 '재료' 찾아내기 (Iteration)
# 목표: 각 샘플에서 핵심 정보(질문, 배경, 정답)를 추출하고, 메타정보를 읽는 연습.
# ----------------------------------------------------------------------
print("\n" + "="*80)
print("💡 [실습 1] 데이터 구조 분석가: 핵심 정보를 파헤치기 (Iteration Practice)")
print("="*80)

for i, sample in enumerate(sample_data_list):
    print(f"\n⭐ [샘플 {i+1}번] 🔎 메타정보 읽기:")
    
    # 메타데이터(metadata)가 중첩된 딕셔너리 안에 있으므로, 안전하게 접근합니다.
    metadata = sample.get("metadata", {})
    source = metadata.get("source", "N/A")
    chapter = metadata.get("chapter", "N/A")
    article_id = metadata.get("article_id", "N/A")
    
    # 법률 문서를 다루는 만큼, 이 정보들을 함께 출력해 보는 것이 중요합니다!
    print(f"  📄 출처/조항 정보: {source} | 장(Chapter): {chapter} | 조항 ID: {article_id}")
    
    # 핵심 요소 출력
    instruction = sample.get("instruction", "N/A")
    input_text = sample.get("input", "N/A")
    output_text = sample.get("output", "N/A")
    
    # 📝 주석: 법률 문서는 'instruction'이 질문, 'input'이 맥락, 'output'이 답변입니다.
    print(f"  ❓ 지시사항 (Instruction): {instruction[:50]}...")
    print(f"  📜 배경 정보 (Input Context): {input_text[:50]}...")
    print(f"  ✅ 목표 결과 (Desired Output): {output_text[:50]}...")
    
# ----------------------------------------------------------------------
# 🧠 실습 2: [프롬프트 엔지니어] AI에게 완벽한 질문지를 만들어주기 (Prompt Generation)
# 목표: 여러 데이터 필드를 조합하여 LLM이 최고의 성능을 낼 수 있는 '프롬프트'를 직접 설계합니다.
# ----------------------------------------------------------------------
print("\n\n" + "="*80)
print("💡 [실습 2] 프롬프트 엔지니어: 법률 AI Chatbot 프롬프트 만들기")
print("="*80)

# 이 루프는 하나의 샘플로도 충분히 효과적인 프롬프트 구조를 만들 수 있음을 보여줍니다.
# 우리는 이 구조를 일반화하여 보여주는 데 중점을 둡니다.
print("\n[AI를 위한 최적의 프롬프트 템플릿 구조 (Template) 예시]")
print("====================================================================")

# 1. 배경 정보(Context)를 명시하는 지시문
template = f"""
[역할 정의] 당신은 한국의 법률 및 공정무역 전문 지식을 갖춘 법률 컨설턴트입니다.
[배경 자료 (Context)] 다음 'Input' 자료를 기반으로 분석을 시작하세요.
---
{sample_data_list[0].get("input", "N/A")}
---

[지시 사항 (Instruction)] 위의 배경 자료와 다음의 요청 사항({sample_data_list[0].get("instruction", "N/A")})을 바탕으로, 사용자에게 가장 명확하고 간결한 답변을 제공하시오.

[요구되는 답변 형식]
답변은 'Output' 필드에 있는 내용과 같이, 핵심 요약만 포함해야 합니다. 어조는 객관적이고 법률적이어야 합니다.
"""

print(template)

# ----------------------------------------------------------------------
# 🗂️ 실습 3: [지식 분류가] 메타데이터로 지식 구조화하기 (Metadata Analysis)
# 목표: 데이터의 분산된 메타데이터('chapter', 'article_id')를 활용해 지식의 흐름을 파악하는 방법을 배웁니다.
# ----------------------------------------------------------------------
print("\n\n" + "="*80)
print("💡 [실습 3] 지식 분류가: 메타데이터를 이용해 지식의 지도 그리기")
print("="*80)

# 메타데이터에 있는 '챕터'나 '출처'를 그룹별로 분류할 때 유용합니다.
chapter_counts = {}

print("\n🔎 분석할 챕터별 빈도수 (어떤 주제가 가장 많은가?):")
for sample in sample_data_list:
    metadata = sample.get("metadata", {})
    chapter = metadata.get("chapter")
    
    if chapter and chapter != "N/A":
        # Python의 딕셔너리 업데이트 패턴을 사용해 카운트를 높입니다.
        if chapter in chapter_counts:
            chapter_counts[chapter] += 1
        else:
            chapter_counts[chapter] = 1

# 결과 출력
for chapter, count in chapter_counts.items():
    print(f"  📚 [{chapter}]: {count}개의 조항 데이터가 존재합니다.")

print("\n🎉🎉 축하합니다! 당신은 데이터 분석가이자, AI 프롬프트 엔지니어가 되었어요! 🎉🎉")
print("이 과정은 LLM을 사용하기 전, 데이터를 '가장 효율적으로' 다듬는 가장 중요한 단계입니다!")